# Train Sea Animals Classifier on Google Colab

This notebook installs dependencies, accepts your `archive.zip` dataset (or mounts Google Drive), writes the training script, runs training on GPU, and saves `resnet18_model.pth`. Run cells top-to-bottom.

In [ ]:
# Install required packages
!pip install -q torch torchvision --extra-index-url https://download.pytorch.org/whl/cu118 || true
!pip install -q seaborn scikit-learn matplotlib

In [ ]:
# Check GPU availability
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
try:
  get_ipython().system('nvidia-smi')
except Exception:
  pass

In [ ]:
# Option A: Upload a zip named archive.zip (ImageFolder layout inside)
from google.colab import files
uploaded = files.upload()
if 'archive.zip' in uploaded:
  print('archive.zip uploaded')
# Option B: Mount Google Drive and copy a zip there (uncomment to use)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/path/to/archive.zip ./archive.zip

# If archive.zip present, extract to ./dataset
import os, zipfile
if os.path.exists('archive.zip'):
  with zipfile.ZipFile('archive.zip','r') as z:
    z.extractall('dataset')
  print('Extracted archive.zip -> ./dataset')
else:
  print('No archive.zip found. Upload archive.zip or mount Drive and copy it here.')

In [ ]:
# Write the training script (train.py)
train_code = r'''
import os
import time
import json
from collections import Counter

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

def prepare_data(dataset_path='dataset', batch_size=32):
    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f"Dataset directory '{dataset_path}' not found")

    train_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    full = datasets.ImageFolder(root=dataset_path, transform=train_transform)
    classes = full.classes

    n = len(full)
    train_size = int(0.7 * n)
    val_size = int(0.15 * n)
    test_size = n - train_size - val_size
    train_ds, val_ds, test_ds = random_split(full, [train_size, val_size, test_size])

    # set eval transforms for val/test
    val_ds.dataset.transform = test_transform
    test_ds.dataset.transform = test_transform

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader, classes, full

def build_model(num_classes, device):
    model = models.resnet18(pretrained=False)
    for param in model.parameters():
        param.requires_grad = False
    in_features = model.fc.in_features
    model.fc = nn.Sequential(nn.Linear(in_features, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, num_classes))
    model = model.to(device)
    return model

def train(model, device, train_loader, val_loader, epochs=8, lr=1e-4, out_path='resnet18_model.pth'):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.fc.parameters(), lr=lr)

    best_val_acc = 0.0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = 100.0 * correct / total

        # validation
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                _, preds = outputs.max(1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_loss = val_loss / val_total if val_total > 0 else 0.0
        val_acc = 100.0 * val_correct / val_total if val_total > 0 else 0.0

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        elapsed = time.time() - t0
        print(f"Epoch {epoch+1}/{epochs} — {elapsed:.1f}s — Train loss {train_loss:.4f}, Train acc {train_acc:.2f}% — Val loss {val_loss:.4f}, Val acc {val_acc:.2f}%")

        # save best
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), out_path)
            print(f"Saved best model (val_acc={best_val_acc:.2f}%) to {out_path}")

    return history

def plot_history(history, out_img='train_curves.png'):
    epochs = range(1, len(history['train_loss']) + 1)
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(epochs, history['train_loss'], label='train')
    plt.plot(epochs, history['val_loss'], label='val')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(epochs, history['train_acc'], label='train')
    plt.plot(epochs, history['val_acc'], label='val')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.savefig(out_img, dpi=150, bbox_inches='tight')
    print(f"Saved training curves to {out_img}")

def test(model, device, test_loader, classes):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = outputs.max(1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.numpy().tolist())
    correct = sum(int(p == t) for p, t in zip(all_preds, all_labels))
    acc = 100.0 * correct / max(1, len(all_labels))
    print(f"Test Accuracy: {acc:.2f}%")
    return acc, all_preds, all_labels

def main():
    dataset_path = 'dataset'
    batch_size = 32
    epochs = 8
    out_model = 'resnet18_model.pth'

    print('Preparing data...')
    train_loader, val_loader, test_loader, classes, full = prepare_data(dataset_path, batch_size)
    print(f'Found {len(classes)} classes')

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Using device:', device)

    model = build_model(len(classes), device)

    if os.path.exists(out_model):
        try:
            model.load_state_dict(torch.load(out_model, map_location=device))
            print('Loaded existing model checkpoint — continuing fine-tuning')
        except Exception as e:
            print('Could not load existing checkpoint:', e)

    history = train(model, device, train_loader, val_loader, epochs=epochs, lr=1e-4, out_path=out_model)

    with open('train_history.json', 'w') as f:
        json.dump(history, f)
    print('Saved train_history.json')

    plot_history(history)

    best_model = build_model(len(classes), device)
    best_model.load_state_dict(torch.load(out_model, map_location=device))
    test_acc, preds, labels = test(best_model, device, test_loader, classes)

if __name__ == '__main__':
    main()
'''
with open('train.py','w') as f:
  f.write(train_code)
print('Wrote train.py')

In [ ]:
# Run training (GPU will be used if available). Adjust --epochs/--batch-size as desired.
!python -u train.py --epochs 8 --batch-size 32

In [ ]:
# After training completes, download the model or copy to Drive
from google.colab import files
import os
if os.path.exists('resnet18_model.pth'):
  files.download('resnet18_model.pth')
else:
  print('resnet18_model.pth not found — check training output')